In [ ]:
%pip install --upgrade pip
%pip install tensorflow notebook ipykernel matplotlib numpy pandas scikit-learn seaborn kagglehub

In [ ]:
# RNN Text Generation using Shakespeare Dataset
# 1. Import Libraries
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# 2. Load Dataset
path = keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
)
text = open(path, encoding="utf-8").read().lower()[:200000]
print("Sample Text:")
print(text[:300])

In [ ]:
# 3. Prepare Character Vocabulary
chars = sorted(set(text))
char_to_id = {c: i for i, c in enumerate(chars)}
encoded = np.array([char_to_id[c] for c in text])
print("Total Characters:", len(text))
print("Vocabulary Size:", len(chars))

In [ ]:
# 4. Create Input Sequences
SEQ_LEN = 40
X = []
y = []
for i in range(0, len(encoded) - SEQ_LEN, 3):
    X.append(encoded[i:i + SEQ_LEN])
    y.append(encoded[i + SEQ_LEN])
X = np.array(X)
y = np.array(y)
print("Input Shape:", X.shape)
print("Output Shape:", y.shape)

In [ ]:
# 5. Build RNN Model
model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN,)),
    layers.Embedding(len(chars), 64),
    layers.SimpleRNN(128),
    layers.Dense(len(chars), activation="softmax")
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
# 6. Train Model
history = model.fit(
    X,
    y,
    batch_size=128,
    epochs=15,
    validation_split=0.1
)

In [ ]:
# 7. Plot Accuracy and Loss
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epochs")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 8. Generate Text Function
def generate_text(seed, length=300, temperature=0.5):
    result = seed.lower()
    for _ in range(length):
        sequence = result[-SEQ_LEN:]
        x = np.array([[char_to_id[c] for c in sequence]])
        predictions = model.predict(x, verbose=0)[0]
        predictions = np.log(predictions + 1e-8) / temperature
        predictions = np.exp(predictions - np.max(predictions))
        predictions = predictions / np.sum(predictions)
        next_char = np.random.choice(chars, p=predictions)
        result += next_char
    return result

In [ ]:
# 9. Generate New Text
seed = text[1000:1000 + SEQ_LEN]
print("\nOriginal Text:")
print(seed)
print("\nGenerated Text:")
print(generate_text(seed, length=300, temperature=0.5))

In [ ]:
# 10. Generate Text with Different Temperatures
for temp in [0.2, 0.5, 1.0]:
    print("\nTemperature:", temp)
    print(generate_text(seed, length=200, temperature=temp))
print("\nRNN Text Generation Completed Successfully!")